# TCR–Peptide Pretraining, Bidirectional DMS & TCR3d Contacts Quickstart

This notebook explores [`aurlienplissier/tcr-peptide-dms`](https://www.kaggle.com/datasets/aurlienplissier/tcr-peptide-dms):
1. **Stage-1 Zero-Leakage Repertoire Pretraining (`pretrain/`)**: `234,147` train + `26,016` val paired `CDR3β|CDR3α`–epitope pairs
2. **Stage-2 Deduplicated `EPACT` Splits (`finetune/`)**: `19,882` train + `2,210` val
3. **Peptide-Anchored DMS (`peptide_dms/`, 71 groups)** & **TCR-Anchored CDR3 DMS (`tcr_dms/`, 23 groups · 703 variants)**
4. **TCR3d `<5 Å` Crystal Contacts (`tcr3d_contacts/`)**: `371` paired TCR–pMHC crystal complexes (`101,962` residue pairs, `5,612` `<5 Å` contacts)
5. **12-Model Precomputed Benchmark Scores (`baseline_scores/`)**

Companion code & checkpoints: [AI-SysBio/LogiCA](https://github.com/AI-SysBio/LogiCA/tree/main/applications/tcrlang-esm) · [ydchen/logica-tcr](https://huggingface.co/ydchen/logica-tcr)

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

candidates = [
    Path("/kaggle/input/tcr-peptide-dms"),
    Path("/kaggle/input/datasets/aurlienplissier/tcr-peptide-dms"),
]
ROOT = next((p for p in candidates if p.exists()), None)
if ROOT is None:
    import kagglehub
    ROOT = Path(kagglehub.dataset_download("aurlienplissier/tcr-peptide-dms"))
print("Dataset root:", ROOT)
print(json.dumps(json.loads((ROOT / "dataset_summary.json").read_text()), indent=2))

In [ ]:
# 1. Verify strict 0% pair leakage between Stage-1 pretraining and Expanded TCR-Anchored DMS
pre_tr = pd.read_csv(ROOT / "pretrain/epact_stapler_iedb_train_non_overlap.csv")
tcr_dms = pd.read_csv(ROOT / "tcr_dms/tcr_mutations_combined_test.csv")
pep_dms = pd.read_csv(ROOT / "peptide_dms/epytope_mutation_test.csv")

train_pairs = set(zip(pre_tr["tcr"].str.strip().str.upper(), pre_tr["epitope"].str.strip().str.upper()))
tcr_test_pairs = set(zip(tcr_dms["tcr"].str.strip().str.upper(), tcr_dms["epitope"].str.strip().str.upper()))
print(f"Stage-1 pretrain pairs: {len(pre_tr):,} | Expanded TCR DMS pairs: {len(tcr_dms):,}")
print(f"Exact (TCR, epitope) overlap between pretrain and TCR DMS test: {len(train_pairs & tcr_test_pairs)}")
print("\nTCR-Anchored DMS source breakdown:")
display(tcr_dms.groupby("source_db").agg(variants=("tcr", "count"), groups=("tcr_name", "nunique")))

In [ ]:
# 2. Inspect TCR3d <5 A CDR3-Epitope Crystal Contacts
t3_comp = pd.read_csv(ROOT / "tcr3d_contacts/tcr3d_complexes.csv")
t3_cont = pd.read_parquet(ROOT / "tcr3d_contacts/tcr3d_residue_contacts.parquet")
print(f"TCR3d resolved paired complexes: {len(t3_comp)} | Residue pairs: {len(t3_cont):,} | <5 A contacts: {int(t3_cont['contact_5A'].sum()):,}")
display(t3_comp[["pdb_id", "tcr_complex", "tcr_name", "epitope", "cdr3_beta", "cdr3_alpha", "n_total_contacts_5A"]].head(6))

# Plot 12-model comparison across Peptide-Anchored vs TCR-Anchored DMS
summary12 = pd.read_csv(ROOT / "baseline_scores/table3_summary_all12_models.csv")
fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.scatter(summary12["pep_spearman_mean"], summary12["tcr_spearman_mean"], s=70, color="#4C78A8")
for _, r in summary12.iterrows():
    ax.annotate(str(r["Method"]), (r["pep_spearman_mean"], r["tcr_spearman_mean"]), fontsize=8, xytext=(4, 3), textcoords="offset points")
ax.axhline(0, ls="--", color="gray", lw=0.8)
ax.axvline(0, ls="--", color="gray", lw=0.8)
ax.set_xlabel("Peptide-Anchored DMS Mean Spearman (71 groups)")
ax.set_ylabel("TCR-Anchored CDR3 DMS Mean Spearman (23 groups)")
ax.set_title("Bidirectional TCR-Peptide Mutational Generalization")
plt.tight_layout()
plt.show()